# ¿Qué ediciones de Wikipedia habrá que deshacer? Un asistente en flujo para patrulleros

**Aprendizaje Automático en Series Temporales y Flujos de Datos** · Máster en IA Aplicada (UC3M) · Proyecto de Flujos de Datos

> **Nota para el grupo (borrar antes de entregar).** Los textos de este notebook son un borrador para que sirva de guía:
> reescribidlos con vuestras palabras y completad las conclusiones con los números que salgan al ejecutarlo.
> Las celdas de código están pensadas para ejecutarse de arriba abajo (*Entorno de ejecución → Ejecutar todo*).

## 1. Introducción y objetivo

Cada hora, la Wikipedia en inglés recibe unas **500 ediciones de usuarios anónimos**. Alrededor de **una de cada cuatro acaba siendo revertida**
(deshecha por otro editor), casi siempre porque era vandalismo, un error o no cumplía las normas. Los voluntarios que las revisan (*patrulleros*)
no pueden mirarlas todas, y hoy las ven en el orden en que llegan.

**Idea del proyecto.** Un sistema que, para cada edición que llega, estima en el momento su **riesgo de ser revertida** y ordena la cola de revisión de más a menos
arriesgada. Así el patrullero empieza por donde es más probable que haga falta actuar.

**Objetivo.** Construir y evaluar un clasificador que aprende **en flujo** (una pasada, sin guardar el histórico, con la etiqueta llegando tarde),
y medir su utilidad real: *si el patrullero revisa solo el 10 % de las ediciones, ¿qué fracción de las malas encuentra?*

Preguntas que respondemos:
1. ¿Aporta el **texto** (resumen de edición y título) por encima de las variables numéricas?
2. ¿Qué modelo incremental es mejor y qué cuesta en **tiempo y memoria**?
3. ¿Cómo afecta que la **etiqueta llegue con retraso** (hay que esperar a que alguien revierta)?
4. ¿El modelo **se queda viejo** (concept drift) y hay que hacerle olvidar, o basta con seguir aprendiendo?
5. ¿Qué ganancia práctica da el mejor modelo en una simulación de patrulla y en directo?

## 2. Trabajos similares

- **Detección de vandalismo en Wikipedia.** Es un problema clásico: Potthast et al. (2008) lo plantearon como clasificación con
  características de texto y metadatos. Wikimedia mantiene sus propios modelos de riesgo de reversión (*Revert Risk*, en su plataforma Lift Wing) y la comunidad usa
  bots como ClueBot NG. Esos sistemas se entrenan por lotes y se actualizan cada cierto tiempo.
- **Aprendizaje en flujos de datos.** Árboles de Hoeffding (Domingos y Hulten, 2000), detección de cambios con ADWIN (Bifet y Gavaldà, 2007) y DDM (Gama et al., 2004),
  bosques adaptativos ARF (Gomes et al., 2017) y la visión general de Gama et al. (2014) sobre *concept drift*.
  Todo ello está implementado en la librería **River** (Montiel et al., 2021), que es la que usamos.
- **Lo que aporta este trabajo.** Tratar el problema como un flujo real: predecir en el instante de llegada, aprender cuando la etiqueta llega con retraso,
  medir la utilidad para el patrullero y vigilar si el modelo envejece.

*(Referencias completas al final. Comprobad los datos de cada una antes de entregar.)*

## 3. Datos y flujo de datos

**Origen.** Ediciones **anónimas** (cuentas temporales e IP) de la Wikipedia en inglés, espacio principal (artículos), sin bots, del **8 al 17 de octubre de 2026**,
descargadas con la API `list=recentchanges` (script `descargar_datos.py` del repositorio).
**Etiqueta.** Una edición está *revertida* si la API le añadió la etiqueta `mw-reverted`.

**Cómo lo tratamos como flujo.** Las ediciones se leen en orden de llegada, una a una, y el modelo solo ve el pasado. La etiqueta de una edición **no se conoce cuando llega**:
se le entrega al modelo pasadas unas horas (`RETRASO_H`), como pasaría de verdad (hay que esperar a que otra persona revierta).

In [ ]:
# --- Instalación de River (sin tocar numpy para no romper Colab; si da error, reiniciar el entorno y volver a ejecutar)
import subprocess, sys
try:
    import river
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "river"], check=True)
    import river
print("River", river.__version__)

In [ ]:
import os, re, math, time, json, pickle, datetime as dt, urllib.request
from collections import defaultdict, deque

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score

from river import base, compose, drift, evaluate, metrics, optim, preprocessing
from river import feature_extraction as fx, linear_model, naive_bayes

# ------------------------------------------------------------------ Parámetros del experimento
URL_CSV   = "https://raw.githubusercontent.com/javiacey16/proyecto-series-temporales/main/data/ediciones_wikipedia.csv.gz"
RUTA_CSV  = "data/ediciones_wikipedia.csv.gz"
MUESTRA   = None      # para pruebas rápidas: p. ej. 30000 (None = todo el flujo)
RETRASO_H = 6         # horas hasta que la etiqueta llega al modelo (supuesto; en la sección 7 se prueba otros)
DIAS_AJUSTE = 3       # los 3 primeros días se usan para ajustar; se evalúa desde el día 4
K_PATRULLA  = 0.10    # fracción de ediciones que revisa el patrullero

# ------------------------------------------------------------------ Estilo de las gráficas
COL = {"Texto": "#2b6cb0", "Numérico": "#d97706", "Apilado": "#15803d", "Azar": "#6b7280", "Regla": "#9ca3af"}
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": .25, "axes.spines.top": False,
                     "axes.spines.right": False, "axes.titlesize": 11, "axes.labelsize": 10})

_t0 = time.time()
def tic(msg=""):
    print(f"[{(time.time() - _t0) / 60:5.1f} min] {msg}")

In [ ]:
# ------------------------------------------------------------------ Carga del CSV (del repositorio) y orden temporal
if not os.path.exists(RUTA_CSV):
    os.makedirs(os.path.dirname(RUTA_CSV), exist_ok=True)
    urllib.request.urlretrieve(URL_CSV, RUTA_CSV)

df = pd.read_csv(RUTA_CSV)
df["timestamp"] = pd.to_datetime(df["timestamp"], utc=True)
df["comment"] = df["comment"].fillna("")
df["title"] = df["title"].fillna("")
df = df.sort_values(["timestamp", "revid"]).reset_index(drop=True)
if MUESTRA:
    df = df.head(MUESTRA)

df["ts"] = (df["timestamp"] - pd.Timestamp("1970-01-01", tz="UTC")).dt.total_seconds()   # segundos (UTC)
df["h_rel"] = (df["ts"] - df["ts"].iloc[0]) / 3600                                       # horas desde el inicio
df["dia"] = (df["h_rel"] // 24).astype(int) + 1                                          # día 1, 2, ...

print(f"{len(df):,} ediciones · del {df.timestamp.min():%d-%m %H:%M} al {df.timestamp.max():%d-%m %H:%M} (UTC)")
print(f"Revertidas: {df.revertida.mean():.1%}  ·  ediciones sin resumen: {(df.comment == '').mean():.1%}")
df[["timestamp", "title", "comment", "oldlen", "newlen", "revertida"]].head(5)

**Variables que recibe el modelo para cada edición** (todas calculables en el momento en que llega la edición):

| Grupo | Variable | Qué es |
|---|---|---|
| Texto | `texto` | Resumen de edición + título de la página |
| Tamaño | `dl`, `long` | Cambio de tamaño (con signo y en escala log) y tamaño previo de la página |
| Hora | `h_sin`, `h_cos` | Hora del día (UTC) codificada en círculo |
| Resumen | `sin_comentario`, `seccion`, `long_com`, `mayus`, `deshacer` | Sin resumen, edición de sección (`/* ... */`), longitud, proporción de mayúsculas y si menciona *undid/revert* |
| Estado del flujo | `act24`, `pag_vista` | Ediciones a esa página en las últimas 24 h y si ya se había editado (solo mira el pasado) |

Las banderas `minor` y `new` son constantes en este conjunto de datos, así que no se usan.

In [ ]:
RE_DESHACER = re.compile(r"undid|revert|\brv\b", re.I)
NUM = ["dl", "long", "h_sin", "h_cos", "sin_comentario", "seccion", "long_com", "mayus", "deshacer", "act24", "pag_vista"]


class Extractor:
    """Convierte una edición en el diccionario x que consume River.
    Guarda cuándo se editó cada página recientemente (solo el pasado) para calcular la actividad de la página."""

    def __init__(self):
        self.ultimas = defaultdict(deque)
        self.n = 0

    def __call__(self, ts, titulo, comentario, oldlen, newlen):
        q = self.ultimas[titulo]
        while q and ts - q[0] > 86400:
            q.popleft()
        act = len(q)
        q.append(ts)
        self.n += 1
        if self.n % 50000 == 0:                      # limpieza: la memoria del estado no debe crecer sin límite
            for k in [k for k, v in self.ultimas.items() if not v or ts - v[-1] > 86400]:
                del self.ultimas[k]
        dl = newlen - oldlen
        hora = (ts % 86400) / 3600
        return {
            "texto": f"{comentario} || {titulo}",
            "dl": math.copysign(math.log1p(abs(dl)), dl),
            "long": math.log1p(max(oldlen, 0)),
            "h_sin": math.sin(2 * math.pi * hora / 24), "h_cos": math.cos(2 * math.pi * hora / 24),
            "sin_comentario": float(comentario == ""),
            "seccion": float(comentario.startswith("/*")),
            "long_com": math.log1p(len(comentario)),
            "mayus": sum(c.isupper() for c in comentario) / (len(comentario) + 1),
            "deshacer": float(bool(RE_DESHACER.search(comentario))),
            "act24": math.log1p(act), "pag_vista": float(act > 0),
        }


def flujo(datos, extractor=None):
    """Genera el flujo: (x, y, ts) en orden de llegada. El modelo nunca ve el futuro."""
    ext = extractor if extractor is not None else Extractor()
    for ts, t, c, o, n, y in zip(datos.ts.values, datos.title.values, datos.comment.values,
                                 datos.oldlen.values, datos.newlen.values, datos.revertida.values):
        yield ext(ts, t, c, o, n), bool(y), ts


def como_river(datos):
    """Mismo flujo como pares (x, y) con la hora de llegada dentro de x, para evaluate.progressive_val_score."""
    for (x, y, ts) in flujo(datos):
        x["ts_dt"] = dt.datetime.fromtimestamp(ts, dt.timezone.utc)
        yield x, y


for x, y, ts in flujo(df.head(3)):
    print(y, {k: (round(v, 2) if isinstance(v, float) else v) for k, v in x.items()}, "\n")

## 4. Análisis del flujo

Antes de modelar miramos cómo es el flujo: ritmo de llegada, balance de clases, patrón por hora del día y si el vocabulario cambia.

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(14, 3.6))

# (a) ritmo de llegada
por_hora = df.groupby(df["h_rel"].astype(int)).size()
ax[0].plot(por_hora.index, por_hora.values, color=COL["Texto"], lw=.9)
ax[0].set(title="Ritmo de llegada (ediciones por hora)", xlabel="horas desde el inicio", ylabel="ediciones/hora")
for d_ in range(24, int(df.h_rel.max()), 24):
    ax[0].axvline(d_, color="k", alpha=.08)

# (b) tasa de reversión por hora del día
hora_dia = df.groupby(df.timestamp.dt.hour).revertida.mean()
ax[1].bar(hora_dia.index, hora_dia.values * 100, color=COL["Numérico"])
ax[1].axhline(df.revertida.mean() * 100, color="k", ls="--", lw=.8, label="media")
ax[1].set(title="% revertidas según la hora (UTC)", xlabel="hora del día", ylabel="% revertidas"); ax[1].legend()

# (c) tasa de reversión por día
por_dia = df.groupby("dia").revertida.agg(["size", "mean"])
ax[2].bar(por_dia.index, por_dia["mean"] * 100, color=COL["Apilado"])
ax[2].set(title="% revertidas por día", xlabel="día del flujo", ylabel="% revertidas", xticks=list(por_dia.index))
plt.tight_layout(); plt.show()

print(por_dia.rename(columns={"size": "ediciones", "mean": "% revertidas"}).assign(**{"% revertidas": lambda t: (t["% revertidas"] * 100).round(1)}).T.to_string())
print("\nOjo: el último día baja. La etiqueta se descargó poco después, y a las ediciones más recientes aún no les ha dado tiempo a ser revertidas (censura por la derecha).")

### Cambios en los datos (data drift) de forma exploratoria

Dos comprobaciones sencillas: (1) el **vocabulario nuevo** que va apareciendo en los resúmenes (si siempre aparecen palabras nuevas, el modelo de texto debe seguir aprendiendo);
(2) un detector de cambios de distribución (**KSWIN**, visto en clase) sobre el cambio de tamaño de las ediciones.

In [ ]:
# (1) % de palabras de cada día que no se habían visto en los días anteriores
vistas, nuevo = set(), {}
for d_, g in df.groupby("dia"):
    palabras = [w for c in g.comment.str.lower() for w in re.findall(r"[a-z]{3,}", c)]
    if d_ > 1 and palabras:
        nuevo[d_] = 100 * np.mean([w not in vistas for w in palabras])
    vistas.update(palabras)

# (2) KSWIN sobre el cambio de tamaño (en el orden del flujo)
kswin = drift.KSWIN(alpha=0.005, window_size=100, stat_size=30, seed=42)
alarmas_ks = []
for i, (v, d_) in enumerate(zip(np.sign(df.newlen - df.oldlen) * np.log1p((df.newlen - df.oldlen).abs()), df.dia)):
    kswin.update(float(v))
    if kswin.drift_detected:
        alarmas_ks.append(d_)

fig, ax = plt.subplots(1, 2, figsize=(11, 3.4))
ax[0].plot(list(nuevo.keys()), list(nuevo.values()), "o-", color=COL["Texto"])
ax[0].set(title="% de palabras nunca vistas antes", xlabel="día", ylabel="% de palabras", xticks=list(nuevo.keys()))
conteo = pd.Series(alarmas_ks).value_counts().reindex(range(1, df.dia.max() + 1), fill_value=0)
ax[1].bar(conteo.index, conteo.values, color=COL["Numérico"])
ax[1].set(title="Alarmas de KSWIN por día (cambio de tamaño)", xlabel="día", ylabel="nº alarmas", xticks=list(conteo.index))
plt.tight_layout(); plt.show()
print(f"KSWIN: {len(alarmas_ks)} alarmas en {len(df):,} ediciones")

## 5. Estrategia de aprendizaje y evaluación

**Aprendizaje.** Online, una edición cada vez: se **predice** al llegar y se **aprende** más tarde, cuando llega la etiqueta (`RETRASO_H` horas después).
Es la evaluación *prequential* (*test-then-train*) de clase, pero con etiqueta retrasada, que es lo que ocurre de verdad. Para controlar el retraso
programamos el bucle nosotros (como en `EvaluacionSinProgressiveValScore`); más adelante lo contrastamos con `evaluate.progressive_val_score(..., delay=...)` de River.

**Métricas.**
- **ROC AUC**: calidad del orden que produce el modelo (0,5 = azar). Es la métrica natural porque el producto es un *ranking*, y no depende del umbral ni de que haya un 25 % de positivos.
- **Precisión y recall @10 %**: si en cada hora el patrullero revisa solo el 10 % de las ediciones con más riesgo, qué fracción de las revisadas eran malas y qué fracción de *todas* las malas encuentra.
  Azar: precisión ≈ 25 %, recall = 10 %.
- **Tiempo y memoria**: ediciones por segundo y tamaño del modelo.

**Partición temporal.** Se **ajustan** los hiperparámetros con los 3 primeros días y se **evalúa** solo desde el día 4, así el ajuste no ve los datos de evaluación.

**Referencias (baselines).** Orden al azar y una regla del patrullero que no aprende: *revisar primero las ediciones que más cambian el tamaño de la página*.

In [ ]:
def prequential(modelo, datos, retraso_h=RETRASO_H, congelar_h=None, reiniciar_h=None, fabrica=None):
    """Test-then-train con etiqueta retrasada. Devuelve las puntuaciones (riesgo) de cada edición y el modelo entrenado.
    congelar_h  : deja de aprender de las ediciones posteriores a esa hora (para estudiar el drift)
    reiniciar_h : crea un modelo nuevo cada tantas horas (olvido total)"""
    ext = Extractor()
    pendientes = deque()                                       # (hora a la que llega la etiqueta, x, y, hora de la edición)
    score = np.zeros(len(datos))
    ts0 = datos.ts.iloc[0]
    proximo = reiniciar_h
    t0 = time.perf_counter()
    for i, (x, y, ts) in enumerate(flujo(datos, ext)):
        h = (ts - ts0) / 3600
        while pendientes and pendientes[0][0] <= ts:            # 1) llegan las etiquetas que ya tocan -> aprender
            _, xp, yp, hp = pendientes.popleft()
            if congelar_h is None or hp < congelar_h:
                modelo.learn_one(xp, yp)
        if reiniciar_h and h >= proximo:
            modelo, proximo = fabrica(), proximo + reiniciar_h
        score[i] = modelo.predict_proba_one(x).get(True, 0.5)   # 2) predecir con lo aprendido hasta ahora
        pendientes.append((ts + retraso_h * 3600, x, y, h))     # 3) la etiqueta llegará más tarde
    seg = time.perf_counter() - t0
    return {"score": score, "modelo": modelo, "seg": seg, "eps": len(datos) / seg, "ext": ext, "pendientes": pendientes}


def patrulla(score, y, horas, k=K_PATRULLA):
    """En cada hora el patrullero revisa el k% con más riesgo. Devuelve precisión y recall de esa revisión."""
    t = pd.DataFrame({"h": np.floor(horas).astype(int), "s": score, "y": y})
    orden = t.groupby("h")["s"].rank(ascending=False, method="first")
    tam = t.groupby("h")["s"].transform("size")
    sel = orden <= np.ceil(k * tam)
    return t.loc[sel, "y"].mean(), t.loc[sel, "y"].sum() / t["y"].sum()


def curva_ganancia(score, y, horas, fracs):
    return [patrulla(score, y, horas, f)[1] for f in fracs]


def evaluar(score, desde_h=24 * DIAS_AJUSTE):
    m = (df.h_rel.values >= desde_h)
    y = df.revertida.values[m]
    p, r = patrulla(score[m], y, df.h_rel.values[m])
    return {"AUC": roc_auc_score(y, score[m]), "precisión@10%": p, "recall@10%": r}


# ------------------------------------------------------------------ Referencias
rng = np.random.default_rng(0)
REF = {"Azar": rng.random(len(df)),
       "Regla de tamaño": (df.newlen - df.oldlen).abs().values.astype(float) + rng.random(len(df)) * 1e-6}
tabla_ref = pd.DataFrame({k: evaluar(v) for k, v in REF.items()}).T
tabla_ref

## 6. Modelos

Probamos **tres modelos incrementales**, elegidos para responder a la pregunta "¿aporta el texto?" y por ser los vistos en clase (S4B, S5 y Ensembles):

| Modelo | Qué ve | Cómo | Por qué |
|---|---|---|---|
| **Texto** | resumen + título | `TFIDF` + `ComplementNB` | Naive Bayes es el clasificador de texto de clase; la variante *Complement* está pensada para clases desbalanceadas |
| **Numérico** | variables numéricas | `StandardScaler` + `LogisticRegression` | Modelo lineal de clase; sin texto, es el contraste para medir la aportación del texto |
| **Apilado** | las dos salidas anteriores | una regresión logística combina la puntuación del modelo de texto y la del numérico | *Stacking* de clase: cada modelo mira lo que mejor sabe y el meta-modelo aprende a combinarlos |

El apilado está programado a mano (como el `StackingClassifier` de clase) para que el meta-modelo reciba **solo las puntuaciones** de los dos modelos base y nunca el texto sin procesar.

**Ajuste de hiperparámetros.** Con los 3 primeros días: suavizado y n-gramas del texto, y optimizador del modelo lineal y del meta-modelo (SGD, visto en clase, frente a AdaGrad y Adam, que adaptan el paso de aprendizaje y suelen ser más estables). La puntuación se mide entre los días 2 y 3, y a partir del día 4 no se vuelve a tocar nada.

In [ ]:
def logit(p):
    p = min(max(p, 1e-4), 1 - 1e-4)
    return math.log(p / (1 - p))


def modelo_texto(alpha=0.3, ngram=(1, 2)):
    return fx.TFIDF(on="texto", lowercase=True, ngram_range=ngram, normalize=True) | naive_bayes.ComplementNB(alpha=alpha)


OPTIMIZADORES = {"SGD": optim.SGD, "AdaGrad": optim.AdaGrad, "Adam": optim.Adam}     # visto en clase: SGD; los adaptativos estabilizan el aprendizaje


def modelo_numerico(opt=("Adam", 0.01)):
    return compose.Select(*NUM) | preprocessing.StandardScaler() | linear_model.LogisticRegression(optimizer=OPTIMIZADORES[opt[0]](opt[1]))


class Apilado(base.Classifier):
    """Stacking: el meta-modelo recibe las puntuaciones (logit) de los modelos de texto y numérico.
    Al aprender, usa lo que los modelos base predecían ANTES de actualizarse (como StackingClassifier de River)."""

    def __init__(self, texto, numerico, opt_meta=("Adam", 0.01)):
        self.texto, self.numerico, self.opt_meta = texto, numerico, opt_meta
        self.meta = preprocessing.StandardScaler() | linear_model.LogisticRegression(optimizer=OPTIMIZADORES[opt_meta[0]](opt_meta[1]))

    def _z(self, x):
        return {"logit_texto": logit(self.texto.predict_proba_one(x).get(True, 0.5)),
                "logit_num": logit(self.numerico.predict_proba_one(x).get(True, 0.5))}

    def predict_proba_one(self, x):
        return self.meta.predict_proba_one(self._z(x))

    def learn_one(self, x, y):
        self.meta.learn_one(self._z(x), y)
        self.texto.learn_one(x, y)
        self.numerico.learn_one(x, y)
        return self


def ajustar(fabrica):
    """Prequential sobre los DIAS_AJUSTE primeros días; AUC entre el día 2 y el final del ajuste."""
    sub = df[df.h_rel < 24 * DIAS_AJUSTE].reset_index(drop=True)
    r = prequential(fabrica(), sub)
    m = sub.h_rel.values >= 24
    return roc_auc_score(sub.revertida.values[m], r["score"][m])

In [ ]:
# ------------------------------------------------------------------ Ajuste con los primeros días
def mejor_de(filas, modelo):
    return max((f for f in filas if f["modelo"] == modelo), key=lambda f: f["AUC ajuste"])["_cfg"]


filas = []
for ng in [(1, 1), (1, 2)]:
    for a in [0.03, 0.3, 1.0]:
        filas.append({"modelo": "Texto", "configuración": f"alpha={a}, n-gramas={ng}", "AUC ajuste": ajustar(lambda: modelo_texto(a, ng)), "_cfg": dict(alpha=a, ngram=ng)})
for opt in [("SGD", 0.01), ("AdaGrad", 0.3), ("Adam", 0.01), ("Adam", 0.001)]:
    filas.append({"modelo": "Numérico", "configuración": f"{opt[0]}({opt[1]})", "AUC ajuste": ajustar(lambda: modelo_numerico(opt)), "_cfg": dict(opt=opt)})
MEJOR_TEXTO, MEJOR_NUM = mejor_de(filas, "Texto"), mejor_de(filas, "Numérico")

for opt_m in [("SGD", 0.01), ("Adam", 0.001), ("Adam", 0.01)]:
    filas.append({"modelo": "Apilado", "configuración": f"meta-modelo {opt_m[0]}({opt_m[1]})",
                  "AUC ajuste": ajustar(lambda: Apilado(modelo_texto(**MEJOR_TEXTO), modelo_numerico(**MEJOR_NUM), opt_m)), "_cfg": opt_m})
OPT_META = mejor_de(filas, "Apilado")
tic("ajuste terminado")
print("Elegido -> texto:", MEJOR_TEXTO, "| numérico:", MEJOR_NUM, "| meta-modelo:", OPT_META)
pd.DataFrame(filas).drop(columns="_cfg").round(4)

In [ ]:
# ------------------------------------------------------------------ Ejecución final: todo el flujo, evaluación desde el día 4
FABRICAS = {
    "Texto":    lambda: modelo_texto(**MEJOR_TEXTO),
    "Numérico": lambda: modelo_numerico(**MEJOR_NUM),
    "Apilado":  lambda: Apilado(modelo_texto(**MEJOR_TEXTO), modelo_numerico(**MEJOR_NUM), OPT_META),
}
RES = {}
for nombre, fab in FABRICAS.items():
    RES[nombre] = prequential(fab(), df)
    RES[nombre]["tam_mb"] = len(pickle.dumps(RES[nombre]["modelo"])) / 1e6
    tic(f"{nombre}: {RES[nombre]['eps']:,.0f} ediciones/s")

resumen = pd.DataFrame({n: evaluar(r["score"]) for n, r in RES.items()}).T
resumen = pd.concat([tabla_ref, resumen])
resumen.round(3)

In [ ]:
desde = 24 * DIAS_AJUSTE
fig, ax = plt.subplots(1, 3, figsize=(15, 3.8))

# (a) AUC por día
for n, r in RES.items():
    aucs = [roc_auc_score(df.revertida[df.dia == d_], r["score"][df.dia == d_]) for d_ in range(DIAS_AJUSTE + 1, df.dia.max() + 1)]
    ax[0].plot(range(DIAS_AJUSTE + 1, df.dia.max() + 1), aucs, "o-", color=COL[n], label=n)
ax[0].axhline(0.5, color=COL["Azar"], ls="--", lw=.8, label="azar")
ax[0].set(title="AUC por día (periodo de evaluación)", xlabel="día", ylabel="AUC", xticks=list(range(DIAS_AJUSTE + 1, df.dia.max() + 1))); ax[0].legend(loc="lower right")

# (b) % de las malas encontradas según % revisado
fracs = np.array([.02, .05, .10, .15, .20, .30, .40, .50])
m = df.h_rel.values >= desde
for n, r in RES.items():
    ax[1].plot(fracs * 100, np.array(curva_ganancia(r["score"][m], df.revertida.values[m], df.h_rel.values[m], fracs)) * 100, "o-", color=COL[n], label=n)
reglas = REF["Regla de tamaño"]
ax[1].plot(fracs * 100, np.array(curva_ganancia(reglas[m], df.revertida.values[m], df.h_rel.values[m], fracs)) * 100, "--", color=COL["Regla"], label="regla de tamaño")
ax[1].plot(fracs * 100, fracs * 100, ":", color=COL["Azar"], label="azar")
ax[1].set(title="Malas encontradas según lo revisado", xlabel="% de ediciones revisadas (cada hora)", ylabel="% de las malas encontradas"); ax[1].legend(loc="upper left")

# (c) aportación de cada modelo @10 %
ax[2].bar(resumen.index, resumen["recall@10%"] * 100, color=[COL.get(i, COL["Regla"]) for i in resumen.index])
ax[2].axhline(10, color="k", ls="--", lw=.8)
ax[2].set(title="Recall@10 %: malas encontradas revisando el 10 %", ylabel="% de las malas")
plt.setp(ax[2].get_xticklabels(), rotation=25, ha="right")
plt.tight_layout(); plt.show()

In [ ]:
mejor = resumen.loc[list(RES)].AUC.idxmax()
indiv = resumen.loc[["Texto", "Numérico"]].AUC.max()
print(f"Mejor modelo: {mejor} (AUC {resumen.loc[mejor, 'AUC']:.3f}, recall@10% {resumen.loc[mejor, 'recall@10%']:.1%}).")
print(f"Texto solo {resumen.loc['Texto', 'AUC']:.3f} frente a numérico solo {resumen.loc['Numérico', 'AUC']:.3f}: "
      f"el texto aporta {resumen.loc['Texto', 'AUC'] - resumen.loc['Numérico', 'AUC']:+.3f} de AUC por encima de las variables numéricas.")
print(f"El apilado {'mejora' if resumen.loc['Apilado', 'AUC'] > indiv else 'no mejora'} al mejor modelo individual: {resumen.loc['Apilado', 'AUC'] - indiv:+.3f} de AUC.")
GANADOR = mejor

## 7. El retraso de la etiqueta

Una edición solo se sabe revertida cuando otra persona la revierte, y eso tarda: minutos, horas o días. Hasta entonces el modelo no puede aprender de ella.
En los datos descargados no consta el momento de la reversión, así que **no conocemos el retraso real**; en vez de suponerlo, comprobamos cuánto importa
repitiendo el experimento del mejor modelo con retrasos de 0, 1, 6 y 24 horas.

In [ ]:
filas = []
for h_ret in [0, 1, 6, 24]:
    r = RES[GANADOR] if h_ret == RETRASO_H else prequential(FABRICAS[GANADOR](), df, retraso_h=h_ret)
    filas.append({"retraso (h)": h_ret, **evaluar(r["score"])})
    tic(f"retraso {h_ret} h")
sens = pd.DataFrame(filas).set_index("retraso (h)")
fig, ax = plt.subplots(figsize=(5.5, 3.4))
ax.plot(sens.index, sens["AUC"], "o-", color=COL["Apilado"])
ax.set(title=f"AUC del modelo «{GANADOR}» según el retraso", xlabel="horas hasta recibir la etiqueta", ylabel="AUC", xticks=list(sens.index)); plt.show()
sens.round(3)

## 8. Eficiencia: tiempo y memoria

En un flujo el modelo tiene que ir **más rápido que la llegada de datos** y con **memoria acotada**. Medimos ediciones procesadas por segundo y el tamaño del modelo ya entrenado.
Además contrastamos nuestro bucle con la función oficial de River (`progressive_val_score` con `delay`, `show_time` y `show_memory`) sobre una muestra.

In [ ]:
llegada = len(df) / (df.ts.iloc[-1] - df.ts.iloc[0])
efi = pd.DataFrame({n: {"ediciones/s": r["eps"], "tamaño del modelo (MB)": r["tam_mb"]} for n, r in RES.items()}).T
print(f"Ritmo real de llegada: {llegada:.2f} ediciones/s. Cada modelo procesa entre {efi['ediciones/s'].min():,.0f} y {efi['ediciones/s'].max():,.0f} por segundo.")
efi.round(3)

In [ ]:
# ------------------------------------------------------------------ Contraste con la función oficial de River (muestra del flujo)
muestra = df.head(20000).reset_index(drop=True)
for n, fab in FABRICAS.items():
    print(f"--- {n}")
    evaluate.progressive_val_score(como_river(muestra), fab(), metrics.ROCAUC(), moment="ts_dt",
                                   delay=dt.timedelta(hours=RETRASO_H), print_every=10000, show_time=True, show_memory=True)

In [ ]:
# ------------------------------------------------------------------ ¿Cuánto cuesta el texto? Unigramas frente a unigramas+bigramas
filas = []
for ng in [(1, 1), (1, 2)]:
    r = prequential(modelo_texto(MEJOR_TEXTO["alpha"], ng), df)
    filas.append({"n-gramas": str(ng), **evaluar(r["score"]), "tamaño (MB)": len(pickle.dumps(r["modelo"])) / 1e6, "ediciones/s": r["eps"]})
pd.DataFrame(filas).set_index("n-gramas").round(3)

## 9. Cambios en el flujo: ¿hay que olvidar?

Si el comportamiento de los vándalos o de los revisores cambia (*concept drift*), un modelo que sigue aprendiendo con los datos antiguos puede quedarse atrás.
La decisión práctica es: **¿conviene hacerle olvidar, o basta con seguir aprendiendo?** Lo comprobamos de tres maneras, de la más práctica a la más técnica:

1. **Comparación directa** del mejor modelo si *aprende siempre*, si *se congela* a los 3 días y si *se reinicia* cada 3 días.
2. **Detectores de clase** (ADWIN, DDM, Page-Hinkley) sobre el error del modelo en el periodo de evaluación (antes, el modelo aún está aprendiendo y su error baja por eso, no por un cambio): ¿avisan de algo?
3. **Cambio inyectado**, para comprobar que los detectores sí funcionarían si hubiera un cambio real.

In [ ]:
# ------------------------------------------------------------------ 1) ¿Olvidar o seguir aprendiendo?
VARIANTES = {"aprende siempre": RES[GANADOR],
             "congelado a los 3 días": prequential(FABRICAS[GANADOR](), df, congelar_h=24 * 3),
             "reinicio cada 3 días": prequential(FABRICAS[GANADOR](), df, reiniciar_h=24 * 3, fabrica=FABRICAS[GANADOR])}
tic("variantes de olvido")
dias = range(DIAS_AJUSTE + 1, df.dia.max() + 1)
fig, ax = plt.subplots(figsize=(7, 3.6))
tabla = {}
for nom, r in VARIANTES.items():
    aucs = [roc_auc_score(df.revertida[df.dia == d_], r["score"][df.dia == d_]) for d_ in dias]
    tabla[nom] = aucs
    ax.plot(list(dias), aucs, "o-", label=nom)
ax.set(title=f"AUC por día de «{GANADOR}» según cómo olvida", xlabel="día", ylabel="AUC", xticks=list(dias)); ax.legend(); plt.show()
pd.DataFrame(tabla, index=list(dias)).T.round(3)

In [ ]:
# ------------------------------------------------------------------ 2) Detectores sobre el error del modelo (error binario con umbral fijo)
sc = RES[GANADOR]["score"]; yy = df.revertida.values.astype(int)
umbral = np.quantile(sc[df.h_rel.values < 24 * DIAS_AJUSTE], 0.75)           # marca como "riesgo" el 25 % más alto de los primeros días
error_total = ((sc >= umbral).astype(int) != yy).astype(int)                  # 1 = el modelo se equivocó
ev = df.h_rel.values >= 24 * DIAS_AJUSTE                                      # solo el periodo de evaluación: antes el modelo aún está aprendiendo
error, dia_ev, h_ev = error_total[ev], df.dia.values[ev], df.h_rel.values[ev]


def alarmas(detector, serie):
    out = []
    for i, e in enumerate(serie):
        detector.update(int(e))
        if detector.drift_detected:
            out.append(i)
    return out


DETECTORES = {"ADWIN": lambda: drift.ADWIN(), "DDM": lambda: drift.binary.DDM(), "Page-Hinkley": lambda: drift.PageHinkley(min_instances=30, delta=0.005, threshold=50)}
filas = []
for nom, mk in DETECTORES.items():
    a = alarmas(mk(), error)
    filas.append({"detector": nom, "alarmas": len(a), "primeras (día)": [int(dia_ev[i]) for i in a[:5]]})
print(f"Tasa de error del modelo en el periodo de evaluación (umbral fijo): {error.mean():.1%} · {len(error):,} ediciones")
pd.DataFrame(filas).set_index("detector")

In [ ]:
# ------------------------------------------------------------------ 3) Cambio inyectado: ¿detectan un empeoramiento real?
rng = np.random.default_rng(1)
mitad = len(error) // 2
error_inj = error.copy()
error_inj[mitad:] = np.maximum(error_inj[mitad:], (rng.random(len(error) - mitad) < 0.15).astype(int))   # de repente falla un 15 % más
filas = []
for nom, mk in DETECTORES.items():
    a = alarmas(mk(), error_inj)
    antes = [i for i in a if i < mitad]; despues = [i for i in a if i >= mitad]
    filas.append({"detector": nom, "falsas alarmas antes": len(antes),
                  "retraso en detectar (ediciones)": (despues[0] - mitad) if despues else None,
                  "retraso (horas)": round(h_ev[despues[0]] - h_ev[mitad], 1) if despues else None})
print(f"Error antes del cambio: {error_inj[:mitad].mean():.1%} · después: {error_inj[mitad:].mean():.1%}")
pd.DataFrame(filas).set_index("detector")

## 10. Implementación práctica

Todo lo anterior responde a "¿funciona?". Aquí respondemos a "¿sirve?". Usamos **solo el mejor modelo** y lo ponemos en el lugar del patrullero:
primero con una **simulación sobre los últimos días** (qué ahorraría) y después **en directo** sobre la Wikipedia real, con evaluación diferida cuando se sepa qué ediciones acabaron revertidas.

In [ ]:
# ------------------------------------------------------------------ 10.1 Simulación de patrulla con el mejor modelo (periodo de evaluación)
m = df.h_rel.values >= 24 * DIAS_AJUSTE
y_m, h_m, s_m = df.revertida.values[m], df.h_rel.values[m], RES[GANADOR]["score"][m]
fr = np.linspace(0.01, 1, 100)
g_mod = np.array(curva_ganancia(s_m, y_m, h_m, fr))
g_reg = np.array(curva_ganancia(REF["Regla de tamaño"][m], y_m, h_m, fr))

def a_revisar(g, objetivo):
    return fr[np.argmax(g >= objetivo)]

fig, ax = plt.subplots(figsize=(6.2, 4))
ax.plot(fr * 100, g_mod * 100, color=COL["Apilado"], lw=2, label=f"modelo «{GANADOR}»")
ax.plot(fr * 100, g_reg * 100, "--", color=COL["Regla"], label="regla de tamaño")
ax.plot(fr * 100, fr * 100, ":", color=COL["Azar"], label="orden de llegada / azar")
ax.set(title="Simulación de patrulla (cada hora, se revisa lo más arriesgado primero)", xlabel="% de ediciones revisadas", ylabel="% de las malas encontradas")
ax.legend(loc="upper left"); plt.show()

p10, r10 = patrulla(s_m, y_m, h_m, 0.10)
print(f"Revisando el 10 % de las ediciones de cada hora, el patrullero encuentra el {r10:.0%} de las malas (al azar: 10 %) "
      f"y {p10:.0%} de lo que revisa es malo (al azar: {y_m.mean():.0%}).")
print(f"Para encontrar la mitad de las malas debe revisar el {a_revisar(g_mod, .5):.0%} de las ediciones; con el orden de llegada, el {a_revisar(fr, .5):.0%}.")

In [ ]:
# ------------------------------------------------------------------ 10.2 Cómo se vería la cola de revisión: las 10 ediciones de mayor riesgo del último día
ultimo = df[df.dia == df.dia.max()].copy()
ultimo["riesgo"] = RES[GANADOR]["score"][ultimo.index]
top = ultimo.nlargest(10, "riesgo")[["timestamp", "title", "comment", "riesgo", "revertida"]]
top["riesgo"] = top["riesgo"].round(2)
top["comment"] = top["comment"].str.slice(0, 60)
top

### 10.2 Demostración en directo

El modelo ganador, ya entrenado con todo el histórico, se conecta al flujo en vivo de la Wikipedia (**EventStreams**, `recentchange`) y puntúa cada edición anónima que llega,
con el mismo extractor de variables. En directo **no hay etiqueta**: se muestra la cola de revisión ordenada por riesgo, y **más tarde** (horas después) se consulta la API para saber cuáles
fueron revertidas y evaluar la captura en diferido.

In [ ]:
# ------------------------------------------------------------------ Modelo listo para producción: se pone al día con las etiquetas pendientes
modelo_vivo, ext_vivo = RES[GANADOR]["modelo"], RES[GANADOR]["ext"]
for _, xp, yp, _ in RES[GANADOR]["pendientes"]:
    modelo_vivo.learn_one(xp, yp)

DEMO_SEG = 120          # duración de la captura en directo (segundos)
EN_VIVO = True          # False para saltarse la demo
STREAM_URL = "https://stream.wikimedia.org/v2/stream/recentchange"
HEADERS = {"User-Agent": "ProyectoFlujosDatosUC3M/1.0 (proyecto academico; https://github.com/javiacey16/proyecto-series-temporales)"}
RE_IP = re.compile(r"^(\d{1,3}\.){3}\d{1,3}$|^[0-9a-fA-F:]+:[0-9a-fA-F:]+$")


def es_anonima(usuario):
    return usuario.startswith("~") or bool(RE_IP.match(usuario))


def edicion_valida(ev):
    """Misma población que los datos de entrenamiento: wikipedia en inglés, artículos, sin bots, anónimas."""
    return (ev.get("wiki") == "enwiki" and ev.get("type") in ("edit", "new") and ev.get("namespace") == 0
            and not ev.get("bot") and es_anonima(ev.get("user", "")) and "length" in ev)

In [ ]:
import requests
from IPython.display import clear_output, display

filas_vivo = []
if EN_VIVO:
    inicio, ultimo_pintado = time.time(), 0
    try:
        with requests.get(STREAM_URL, headers=HEADERS, stream=True, timeout=(10, 30)) as r:
            for linea in r.iter_lines():
                if time.time() - inicio > DEMO_SEG:
                    break
                if not linea or not linea.startswith(b"data:"):
                    continue
                try:
                    ev = json.loads(linea[5:])
                except ValueError:
                    continue
                if not edicion_valida(ev):
                    continue
                x = ext_vivo(ev["timestamp"], ev["title"], ev.get("comment", ""), ev["length"].get("old") or 0, ev["length"]["new"])
                filas_vivo.append({"revid": ev["revision"]["new"], "hora": dt.datetime.fromtimestamp(ev["timestamp"], dt.timezone.utc).strftime("%H:%M:%S"),
                                   "title": ev["title"], "comment": ev.get("comment", "")[:60],
                                   "riesgo": round(modelo_vivo.predict_proba_one(x).get(True, 0.5), 3), "ts": ev["timestamp"]})
                if time.time() - ultimo_pintado > 15 and filas_vivo:                  # refresca la cola cada 15 s
                    ultimo_pintado = time.time()
                    clear_output(wait=True)
                    print(f"En directo · {len(filas_vivo)} ediciones puntuadas · {len(filas_vivo) / (time.time() - inicio):.2f} ediciones/s")
                    display(pd.DataFrame(filas_vivo).nlargest(10, "riesgo")[["hora", "title", "comment", "riesgo"]])
    except requests.RequestException as e:
        print("No se pudo conectar al flujo en directo:", e)

demo = pd.DataFrame(filas_vivo)
if len(demo):
    demo.to_csv("demo_en_vivo.csv", index=False)
    clear_output(wait=True)
    print(f"Captura terminada: {len(demo)} ediciones puntuadas en {DEMO_SEG} s (guardadas en demo_en_vivo.csv)")
    display(demo.nlargest(10, "riesgo")[["hora", "title", "comment", "riesgo"]])

**Evaluación diferida.** Pasadas unas horas (mejor un día), la siguiente celda pregunta a la API cuáles de las ediciones capturadas acabaron revertidas y calcula las mismas métricas que antes.

In [ ]:
# ------------------------------------------------------------------ Ejecutar HORAS DESPUÉS de la captura en directo
def etiquetas_api(revids):
    """Consulta la API de Wikipedia: ¿tiene la etiqueta mw-reverted cada revisión?"""
    out = {}
    for i in range(0, len(revids), 50):
        lote = revids[i:i + 50]
        r = requests.get("https://en.wikipedia.org/w/api.php", headers=HEADERS, timeout=60,
                         params={"action": "query", "prop": "revisions", "rvprop": "ids|tags", "revids": "|".join(map(str, lote)),
                                 "format": "json", "formatversion": 2})
        for pag in r.json().get("query", {}).get("pages", []):
            for rev in pag.get("revisions", []):
                out[rev["revid"]] = "mw-reverted" in rev.get("tags", [])
        time.sleep(1)
    return out


if os.path.exists("demo_en_vivo.csv"):
    demo = pd.read_csv("demo_en_vivo.csv")
    etiq = etiquetas_api(demo.revid.tolist())
    demo["revertida"] = demo.revid.map(etiq)
    ok = demo.dropna(subset=["revertida"]).copy(); ok["revertida"] = ok.revertida.astype(bool)
    if ok.revertida.nunique() == 2:
        n10 = int(np.ceil(.10 * len(ok))); top10 = ok.nlargest(n10, "riesgo")
        print(f"{len(ok)} ediciones con etiqueta · {ok.revertida.mean():.0%} revertidas · AUC en directo: {roc_auc_score(ok.revertida, ok.riesgo):.3f}")
        print(f"Revisando el 10 % con más riesgo: {top10.revertida.mean():.0%} eran malas y se encuentra el {top10.revertida.sum() / ok.revertida.sum():.0%} de todas las malas.")
    else:
        print("Todavía no hay suficientes reversiones para evaluar: volved a ejecutar esta celda más tarde.")
else:
    print("No hay captura en directo guardada (demo_en_vivo.csv).")

## 11. Conclusiones

> **Completad esta sección con vuestras palabras tras ejecutar el notebook.** Guía de lo que debe quedar contestado, con los números de las tablas:

1. **El texto aporta**: diferencia de AUC entre el modelo de texto y el numérico (sección 6) y qué parte de esa mejora se mantiene en el apilado.
2. **Mejor modelo** y cuánto gana frente al azar y frente a la regla simple (recall@10 %, sección 6 y 10.1).
3. **Retraso de etiqueta**: cuánto empeora el modelo al pasar de 0 a 24 horas (sección 7).
4. **Coste**: ediciones por segundo frente al ritmo real de llegada, y tamaño de los modelos; qué supone usar bigramas (sección 8).
5. **Drift**: si olvidar mejora o empeora frente a seguir aprendiendo, qué dicen los detectores y si detectan el cambio inyectado (sección 9).
6. **Utilidad práctica**: porcentaje de ediciones que debe revisar el patrullero para encontrar la mitad de las malas, y qué pasó en directo (sección 10).

### Limitaciones
- **Retraso de la etiqueta desconocido**: no tenemos la hora de la reversión; se estudió la sensibilidad (sección 7), pero habría que medirlo.
- **Censura por la derecha**: las ediciones más recientes del CSV tuvieron menos tiempo para ser revertidas, lo que infraestima las positivas del último día.
- **Etiqueta imperfecta**: `mw-reverted` no distingue vandalismo de ediciones revertidas por otros motivos, y una edición mala puede no revertirse nunca.
- **Alcance**: solo Wikipedia en inglés, artículos y ediciones anónimas, durante diez días. Algunos resúmenes de «deshacer» son reversiones de otras ediciones.
- **Evaluación en directo**: muestra pequeña y de pocas horas.

### Mejoras posibles
Historial por usuario (el CSV no incluye el usuario), uso de la hora real de reversión, embeddings de texto (visto en clase: `all-MiniLM-L6-v2` + Hoeffding Tree o Naive Bayes Gaussiano) y ensembles con ADWIN (`ADWINBagging`, `ARFClassifier`), a costa de más tiempo y memoria.

### Uso de herramientas de IA generativa
> Completad con lo que hayáis hecho de verdad. Ejemplo: «Se ha usado Claude (Anthropic) como apoyo para depurar y estructurar el código y para revisar textos redactados por el grupo; las decisiones de diseño, los experimentos y las conclusiones son del grupo.»

### Referencias
- Bifet, A. y Gavaldà, R. (2007). *Learning from time-changing data with adaptive windowing.* SIAM International Conference on Data Mining.
- Domingos, P. y Hulten, G. (2000). *Mining high-speed data streams.* ACM SIGKDD.
- Gama, J., Medas, P., Castillo, G. y Rodrigues, P. (2004). *Learning with drift detection.* SBIA.
- Gama, J., Žliobaitė, I., Bifet, A., Pechenizkiy, M. y Bouchachia, A. (2014). *A survey on concept drift adaptation.* ACM Computing Surveys.
- Gomes, H. M. et al. (2017). *Adaptive random forests for evolving data stream classification.* Machine Learning.
- Montiel, J. et al. (2021). *River: machine learning for streaming data in Python.* Journal of Machine Learning Research.
- Potthast, M., Stein, B. y Gerling, R. (2008). *Automatic vandalism detection in Wikipedia.* ECIR.
- Wikimedia Foundation: API MediaWiki (`recentchanges`), EventStreams y modelos Revert Risk (Lift Wing).